# P1. Demanda diaria de salidas por estación

Exploración del panel estación-día y de los resultados del experimento base de P1.
Requiere haber corrido antes:

```
python -m mibici.dataset --start-year 2025 --start-month 1 --end-year 2026 --end-month 9
python -m mibici.features
python -m mibici.modeling.train_p1
```

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mibici import features as fe
from mibici.config import CONFIG
from mibici.modeling import train_p1 as t1

nombres = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
panel = pd.read_parquet(CONFIG.interim_data_dir / "p1_panel_diario.parquet")
resumen = json.loads((CONFIG.interim_data_dir / "p1_panel_diario.json").read_text())
resumen

## Calidad del panel

El conteo diario del sistema completo sirve para detectar días con datos faltantes o atípicos: un
día sin ninguna salida en todas las estaciones casi seguro es un hueco de los archivos, no demanda
cero, y los días con una caída fuerte (festivos, eventos, cierres) distorsionan los rezagos de los
días siguientes.

In [ ]:
diario = panel.groupby("fecha")["salidas"].agg(total="sum", estaciones="size")
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
diario["total"].plot(ax=ax[0], lw=1, title="Salidas por día (todas las estaciones)")
diario["estaciones"].plot(ax=ax[1], lw=1, title="Estaciones en el panel")
for a in ax:
    for corte in ("val_inicio", "test_inicio"):
        a.axvline(pd.Timestamp(t1.CORTES[corte]), color="gray", ls="--")
plt.tight_layout()
plt.show()
print("Días sin salidas en todo el sistema:", int((diario["total"] == 0).sum()))
menores = diario.nsmallest(10, "total")
menores.assign(dia=[nombres[i] for i in menores.index.dayofweek])

## Patrón semanal y festivos

Salidas del día relativas al nivel reciente de la estación (`objetivo / media_28d`). Un valor de 1
significa un día típico; el dato usa solo estaciones-día elegibles.

In [ ]:
df = fe.construir_features(panel)
elegibles = df[df["elegible"]]
relativo = elegibles["objetivo"] / elegibles["media_28d"]
por_dia = relativo.groupby(elegibles["dia_semana"]).median()
por_dia.index = nombres
por_dia.plot.bar(figsize=(8, 3.5), title="Mediana de salidas relativas al nivel reciente")
plt.axhline(1, color="gray", ls="--")
plt.show()
relativo.groupby(elegibles["festivo"]).median().rename({0: "día normal", 1: "festivo"}).round(3)

## Volumen por estación

Las estaciones se agrupan en terciles de salidas diarias promedio calculados solo con el
entrenamiento; las que no tienen filas de entrenamiento se reportan como nuevas.

In [ ]:
partes = t1.particionar(elegibles)
nivel = partes["train"].groupby("estacion")["objetivo"].mean()
ax = nivel.plot.hist(bins=40, figsize=(9, 3.5), title="Salidas diarias promedio por estación (entrenamiento)")
for corte in nivel.quantile([1 / 3, 2 / 3]):
    ax.axvline(corte, color="gray", ls="--")
plt.show()
nivel.describe().round(1)

## Elegibilidad por mes

Un día entra al modelo si la estación tiene 29 días de historia y alguna salida en los últimos 28
días disponibles; las dos reglas usan solo información anterior a D-1.

In [ ]:
estado = np.select([df["sin_historia"], df["inactiva"]], ["sin_historia", "inactiva"], "elegible")
pd.crosstab(df["fecha"].dt.to_period("M"), estado)

## Resultados del experimento (leídos del archivo que escribe train_p1)

In [ ]:
res = json.loads((CONFIG.reports_dir / "figures" / "p1_resultados.json").read_text())
print("Seleccionado en validación:", res["seleccionado"])
columnas = [
    "val_mae", "test_mae", "test_rmse", "test_wape", "test_bias",
    "test_mae_baja", "test_mae_media", "test_mae_alta", "test_mae_nuevas",
]
pd.DataFrame(res["resultados"]).T.reindex(columns=columnas).round(3)

In [ ]:
print("Parámetros elegidos:", res["elegidos"])
for candidato, filas in res["rejilla"].items():
    print(candidato)
    display(pd.DataFrame(filas).round(4))

## Error por mes y por día de la semana en la prueba

Se reentrenan los candidatos con los parámetros elegidos (misma semilla que el experimento) para
comparar el WAPE mes a mes; sirve para ver si el error se concentra en algún periodo.

In [ ]:
train, test = partes["train"], partes["test"]
real = test["objetivo"].to_numpy()
poisson = t1.ajustar_poisson(train, res["elegidos"]["poisson"]["alpha"])
hgb = t1.ajustar_hgb(train, res["elegidos"]["hgb_poisson"], 29)
pred = {
    "semana_previa": test["baseline_semana_previa"].to_numpy(),
    "persistencia": test["baseline_persistencia"].to_numpy(),
    "poisson": poisson.predict(test[fe.FEATURES]),
    "hgb_poisson": hgb.predict(test[fe.FEATURES]),
}
error = pd.DataFrame({k: np.abs(v - real) for k, v in pred.items()}, index=test.index)
mes = test["fecha"].dt.to_period("M")
wape_mes = error.groupby(mes).sum().div(test.groupby(mes)["objetivo"].sum(), axis=0)
wape_mes.plot(marker="o", figsize=(9, 3.5), title="WAPE por mes de prueba")
plt.show()
wape_mes.round(3)

In [ ]:
dia = test["dia_semana"]
wape_dia = error.groupby(dia).sum().div(test.groupby(dia)["objetivo"].sum(), axis=0)
wape_dia.index = nombres
wape_dia.round(3)

## Coeficientes del Poisson

`efecto_multiplicativo` es exp(coeficiente): cuánto multiplica la media esperada un aumento de una
desviación estándar en la variable (conteos en log1p) o el cambio de categoría (día, festivo).

In [ ]:
pd.read_csv(CONFIG.reports_dir / "figures" / "p1_poisson_coeficientes.csv").round(3)